# 05 - End-to-End Quantum Adaptive Beamforming
Integrates simulated signal covariance with quantum linear solver to generate and compare classical vs quantum beam patterns.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from eigenwall.signal_model import ArrayConfig, steering_vector, generate_received_signal, generate_sinusoid
from eigenwall.covariance import sample_covariance, regularize_covariance
from eigenwall.beamforming import mvdr_weights, beam_pattern, quantum_weights_to_beamforming
from eigenwall.quantum.hhl_solver import quantum_linear_solver
from eigenwall.metrics import state_fidelity, null_depth_db

config = ArrayConfig(n_elements=4)
s = generate_sinusoid(5.0, 1000)
i = generate_sinusoid(18.0, 1000, amplitude=1.8)
X = generate_received_signal(config, 20.0, s, [(-30.0, i)], noise_power=0.8, random_seed=42)
R_raw = regularize_covariance(sample_covariance(X), epsilon=1.0)
R = R_raw * (2.0 / np.linalg.eigvalsh(R_raw)[-1])
a_target = steering_vector(4, 20.0)

w_c = mvdr_weights(R, a_target)
res = quantum_linear_solver(R, a_target, phase_qubits=4)
w_q = quantum_weights_to_beamforming(res['solution_state'], a_target)

angles = np.linspace(-90, 90, 361)
_, _, db_c = beam_pattern(w_c, angles)
_, _, db_q = beam_pattern(w_q, angles)

plt.figure(figsize=(8, 4))
plt.plot(angles, db_c, label='Classical MVDR')
plt.plot(angles, db_q, '--', label='Quantum HHL')
plt.title('Classical vs Quantum MVDR Beam Pattern')
plt.legend()
plt.show()